In [55]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('Dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Dataset\gender_submission.csv
Dataset\test.csv
Dataset\train.csv


In [56]:
train = pd.read_csv(r'Dataset\train.csv')
test = pd.read_csv(r'Dataset\test.csv')
train.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


## Clean the Data. 

- see how many null values does the data have: <br>
```

print('Missing values Percentage: \n\n', round(df.isnull().sum().sort_values(ascending=False)/len(df)*100,1))

In [57]:
print('Missing values Percentage: \n\n', round (train.isnull().sum().sort_values(ascending=False)/len(train)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))


Missing values Percentage: 

 Cabin          77.1
Age            19.9
Embarked        0.2
PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
dtype: float64
Missing values Percentage: 

 Cabin          78.2
Age            20.6
Fare            0.2
PassengerId     0.0
Pclass          0.0
Name            0.0
Sex             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Embarked        0.0
dtype: float64


we see that the titles can be category of guesing the age of a person. like the title master. which can only be used for boys. and now we are going to extract the title and use it as one of hte methods filling up the age with mean values base on class, sex, parent (if they have or none) and title 

In [58]:
from sklearn.model_selection import train_test_split


y = train.Survived



X_train, X_valid, y_train, y_valid = train_test_split(train, y, train_size=0.8, test_size=0.2, random_state= 0)


X_train['Titles'] = X_train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
X_valid['Titles'] = X_valid['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Titles'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)


In [59]:
X_train.shape

(712, 13)

In [60]:
display(X_train.groupby(['Sex', 'Pclass', 'Parch', 'Titles'])['Age'].mean())

Sex     Pclass  Parch  Titles  
female  1       0      Countess    33.000000
                       Lady        48.000000
                       Miss        36.400000
                       Mlle        24.000000
                       Mme         24.000000
                       Mrs         39.470588
                1      Miss        21.500000
                       Mrs         47.375000
                2      Miss        19.888889
                       Mrs         30.500000
        2       0      Miss        29.633333
                       Mrs         34.055556
                       Ms          28.000000
                1      Miss        13.200000
                       Mrs         34.500000
                2      Miss        11.600000
                       Mrs         34.000000
                3      Mrs         39.000000
        3       0      Miss        21.575758
                       Mrs         29.142857
                1      Miss         5.159091
                       

we need to group titles like captain, col, major as one in order to remove sparsity. where a model learns less to feature that appears rarely. 

In [61]:
TitleDict = {"Capt": "Officer","Col": "Officer","Major": "Officer","Jonkheer": "Royalty", \
             "Don": "Royalty", "Sir" : "Royalty","Dr": "Royalty","Rev": "Royalty", \
             "Countess":"Royalty", "Mme": "Mrs", "Mlle": "Miss", "Ms": "Mrs","Mr" : "Mr", \
             "Mrs" : "Mrs","Miss" : "Miss","Master" : "Master","Lady" : "Royalty"}

X_train['Titles'] = X_train.Titles.map(TitleDict)
X_valid['Titles'] = X_valid.Titles.map(TitleDict)
test['Titles'] = test.Titles.map(TitleDict)

In [62]:
display(X_valid.groupby(['Sex','Titles', 'Pclass', 'Parch'])['Age'].mean())

Sex     Titles   Pclass  Parch
female  Miss     1       0        31.777778
                         1        22.000000
                         2        24.500000
                 2       0        33.750000
                         1         3.000000
                         2         7.000000
                 3       0        22.100000
                         1         0.750000
                         2         7.333333
        Mrs      1       0        36.250000
                         1        44.666667
                 2       0        31.600000
                         1        32.000000
                         2        24.000000
                 3       0        40.000000
                         1        36.333333
                         2        28.000000
                         4        29.000000
                         5        41.000000
        Royalty  1       0        49.000000
male    Master   2       1         3.000000
                 3       1         5.473333
 

In [63]:
# GROUP AND FIND THE AVERAGE 

master_means = X_train.groupby(['Sex','Titles', 'Pclass', 'Parch'])['Age'].mean().round(2)
general_means = X_train.groupby(['Sex', 'Pclass', 'Parch'])['Age'].mean().round(2)
global_means = X_train['Age'].mean().round(2)

def FindingAverage(dataset, master, general, global_means):
    # Match the current dataset's rows to the master lookup map indices
    master_means = dataset.set_index(['Sex', 'Titles', 'Pclass', 'Parch']).index.map(master).to_series()
    master_means.index = dataset.index # Keep indices perfectly aligned with the current dataset
    
    # Match the current dataset's rows to the general lookup map indices
    general_means = dataset.set_index(['Sex', 'Pclass', 'Parch']).index.map(general).to_series()
    general_means.index = dataset.index

    dataset['Age'] = dataset['Age'].fillna(master_means)
    dataset['Age'] = dataset['Age'].fillna(general_means)
    dataset['Age'] = dataset['Age'].fillna(global_means)

    return dataset


X_train = FindingAverage(X_train, master_means,general_means, global_means)
X_valid = FindingAverage(X_valid, master_means,general_means, global_means)
test = FindingAverage(test, master_means,general_means, global_means)

In [64]:
train.sample(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
358,359,1,3,"McGovern, Miss. Mary",female,NaN,0,0,330931,7.8792,NaN,Q
656,657,0,3,"Radeff, Mr. Alexander",male,NaN,0,0,349223,7.8958,NaN,S
425,426,0,3,"Wiseman, Mr. Phillippe",male,NaN,0,0,A/4. 34244,7.2500,NaN,S
477,478,0,3,"Braund, Mr. Lewis Richard",male,29.0,1,0,3460,7.0458,NaN,S


In [65]:
print('Missing values Percentage: \n\n', round (X_train.isnull().sum().sort_values(ascending=False)/len(X_train)*100,1))
print('Missing values Percentage: \n\n', round (X_valid.isnull().sum().sort_values(ascending=False)/len(X_valid)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))

Missing values Percentage: 

 Cabin          77.1
Embarked        0.3
PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Titles          0.0
dtype: float64
Missing values Percentage: 

 Cabin          77.1
PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Embarked        0.0
Titles          0.0
dtype: float64
Missing values Percentage: 

 Cabin          78.2
Fare            0.2
Titles          0.2
PassengerId     0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Embarked        0.0
dtype: float64


lets fill up the remaining missing parts

In [66]:
X_train['Embarked'] = X_train['Embarked'].fillna('U')
X_valid['Embarked'] = X_valid['Embarked'].fillna('U')

fillFare = test['Fare'].mean().round(2)
test['Fare'] = test['Fare'].fillna(fillFare)
X_train = X_train.drop(['Ticket', 'Cabin', 'PassengerId', 'Titles', 'Name'], axis=1)
X_valid = X_valid.drop(['Ticket', 'Cabin', 'PassengerId', 'Titles', 'Name'], axis=1)
test = test.drop(['Ticket', 'Cabin', 'PassengerId', 'Titles', 'Name'], axis=1)
X_train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
140,0,3,female,32.5,0,2,15.2458,C
439,0,2,male,31.0,0,0,10.5000,S
817,0,2,male,31.0,1,1,37.0042,C
378,0,3,male,20.0,0,0,4.0125,C
491,0,3,male,21.0,0,0,7.2500,S


In [67]:
print('Missing values Percentage: \n\n', round (X_train.isnull().sum().sort_values(ascending=False)/len(X_train)*100,1))
print('Missing values Percentage: \n\n', round (X_valid.isnull().sum().sort_values(ascending=False)/len(X_valid)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))


Missing values Percentage: 

 Survived    0.0
Pclass      0.0
Sex         0.0
Age         0.0
SibSp       0.0
Parch       0.0
Fare        0.0
Embarked    0.0
dtype: float64
Missing values Percentage: 

 Survived    0.0
Pclass      0.0
Sex         0.0
Age         0.0
SibSp       0.0
Parch       0.0
Fare        0.0
Embarked    0.0
dtype: float64
Missing values Percentage: 

 Pclass      0.0
Sex         0.0
Age         0.0
SibSp       0.0
Parch       0.0
Fare        0.0
Embarked    0.0
dtype: float64


In [68]:
X_train.shape, X_valid.shape, test.shape

((712, 8), (179, 8), (418, 7))

# XGBoost

for this model we are going to use an XGClassifier model. XGboost training compared to a random forest training Sequentially. It builds one shallow tree, calculates its errors, and builds the next tree specifically to correct those mistakes.